# Intermediate Pandas

**Estimated time:** 90-120 minutes
**Prerequisites:** Basic Python, familiarity with Series and DataFrame

## Learning Goals

| # | Topic |
|---|-------|
| 1 | Multi-level indexing and `xs` slicing |
| 2 | `groupby` — transform, filter, and agg together |
| 3 | Merging and joining DataFrames |
| 4 | Pivot tables and `melt` (reshaping) |
| 5 | String methods (`.str`) and categorical dtype |
| 6 | Time series — resampling and rolling windows |
| 7 | `apply` and vectorized alternatives |
| 8 | `pipe` for method chaining |
| 9 | Essential Data I/O — `read_csv`, `read_excel`, `to_csv`, JSON |
| 10 | Missing data and cleaning — `dropna`, `fillna`, `replace`, `rename` |
| 11 | Sorting, ranking, and exploration — `sort_values`, `rank`, `corr` |
| 12 | Binning, dummies, and sampling — `pd.cut`, `pd.qcut`, `get_dummies` |
| 13 | Concatenation and reshaping — `pd.concat`, `stack`, `unstack` |
| 14 | Advanced groupby — iteration, `pd.crosstab`, quantile buckets |
| 15 | Time series: datetime parsing and `ewm` |
| 16 | pandas plotting — line, bar, histogram, scatter |

---

### Quick Reference

```python
df.set_index(['col1','col2']).sort_index()   # MultiIndex
df.xs(key, level='name')                     # inner-level slice
df.groupby('col').agg({'c': ['mean','sum']}) # agg
df.groupby('col').transform('mean')          # transform (keeps shape)
pd.merge(left, right, on='key', how='left')  # merge
df.pivot_table(values, index, columns, aggfunc)
df['col'].str.extract(r'pattern')            # regex extract
df.resample('ME').sum(); df.rolling(30).mean()
df.pipe(func)                                # method chaining
pd.read_csv(path, index_col=0, dtype={...})  # I/O
df.dropna(how='any'); df.fillna(0); df.ffill()
df.sort_values(by, ascending); s.rank()
pd.cut(s, bins); pd.qcut(s, q); pd.get_dummies(df)
pd.concat([df1, df2], keys=[...]); df.stack(); df.unstack()
pd.crosstab(rows, cols, values, aggfunc)
ts.ewm(span=12).mean(); ts.resample('W').interpolate()
df.plot.bar(); s.plot.hist(bins=12)
```

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 100)
print('pandas', pd.__version__)

---
## Section 1 — Multi-Level Indexing

A **MultiIndex** lets you represent hierarchical data (e.g. company → year → quarter) without repeating columns.

Key methods:
- `set_index([col1, col2])` — build the index
- `df.loc[(level0_val, level1_val)]` — label-based selection
- `df.xs(key, level=name)` — cross-section across any level
- `df.reset_index()` — flatten back to regular columns

In [ ]:
# Build a small claims dataset with hierarchical structure
np.random.seed(42)
companies = ['Alpha', 'Beta', 'Gamma']
lines = ['Auto', 'Property', 'Liability']
years = [2021, 2022, 2023]

rows = []
for co in companies:
    for ln in lines:
        for yr in years:
            rows.append({
                'company': co,
                'line': ln,
                'year': yr,
                'premium': np.random.randint(500_000, 5_000_000),
                'claims': np.random.randint(100_000, 3_000_000),
            })

df = pd.DataFrame(rows)
df['loss_ratio'] = df['claims'] / df['premium']
print(df.shape)
df.head(6)

In [ ]:
# Set a two-level index
dfm = df.set_index(['company', 'line']).sort_index()
dfm.head(9)

In [ ]:
# Accessing with .loc — tuple selects both levels
dfm.loc[('Alpha', 'Auto')]

In [ ]:
# .xs lets you slice on an inner level without specifying the outer
# Get all 'Auto' rows across every company
dfm.xs('Auto', level='line')

In [ ]:
# EXERCISE 1: Use .xs to retrieve all rows for year 2022
# Hint: first you need 'year' in the index — try adding it to set_index

dfm3 = df.set_index(['company', 'line', 'year']).sort_index()
# YOUR CODE HERE
# result = dfm3.xs(...)
# result

---
## Section 2 — GroupBy: agg, transform, filter

Three distinct use cases:

| Method | Returns | Use when |
|--------|---------|----------|
| `.agg()` | One row per group (reduced) | Summary statistics |
| `.transform()` | Same shape as input | Add group-level stats as columns |
| `.filter()` | Subset of original rows | Keep/drop entire groups |

Named aggregations with `pd.NamedAgg` keep output column names clean.

In [ ]:
# Multi-function agg — returns a MultiIndex column
summary = df.groupby('line').agg(
    total_premium=('premium', 'sum'),
    total_claims=('claims', 'sum'),
    avg_lr=('loss_ratio', 'mean'),
    std_lr=('loss_ratio', 'std'),
    n=('year', 'count'),
)
summary

In [ ]:
# transform: add a column showing each row's deviation from its group mean
df['line_avg_lr'] = df.groupby('line')['loss_ratio'].transform('mean')
df['lr_vs_line_avg'] = df['loss_ratio'] - df['line_avg_lr']

# Companies where LR is worse than the line average show positive deviation
df[['company', 'line', 'year', 'loss_ratio', 'line_avg_lr', 'lr_vs_line_avg']].head(9)

In [ ]:
# filter: keep only lines where the average loss ratio > 0.55
high_lr = df.groupby('line').filter(lambda g: g['loss_ratio'].mean() > 0.55)
print('Lines kept:', high_lr['line'].unique())
high_lr.shape

In [ ]:
# EXERCISE 2:
# a) For each (company, year), compute total premium and total claims
# b) Add a column 'rank_within_year' that ranks companies by total premium within each year
#    Hint: groupby('year')['total_premium'].rank(ascending=False)

# YOUR CODE HERE

---
## Section 3 — Merging and Joining

```
pd.merge(left, right, on=key, how='left|right|inner|outer')
df.join(other, on=key)   # index-based shorthand
```

| Join type | Keeps |
|-----------|-------|
| `inner`   | Only rows with matches in **both** |
| `left`    | All left rows; NaN where no right match |
| `right`   | All right rows; NaN where no left match |
| `outer`   | All rows from both sides |

**Diagnostic tip:** check `df.merge(..., indicator=True)['_merge'].value_counts()` to spot unmatched rows.

In [ ]:
# Company metadata table
meta = pd.DataFrame({
    'company': ['Alpha', 'Beta', 'Gamma', 'Delta'],  # Delta has no claims data
    'region': ['Northeast', 'Southeast', 'Midwest', 'West'],
    'founded': [1982, 1995, 2003, 2010],
})

# Aggregate df to company level first
co_summary = df.groupby('company', as_index=False).agg(
    total_premium=('premium', 'sum'),
    total_claims=('claims', 'sum'),
)
co_summary['combined_lr'] = co_summary['total_claims'] / co_summary['total_premium']
co_summary

In [ ]:
# Inner join — only Alpha, Beta, Gamma (Delta has no claims data)
inner = pd.merge(co_summary, meta, on='company', how='inner')
inner

In [ ]:
# Outer join with indicator — Delta appears with NaN financials
outer = pd.merge(co_summary, meta, on='company', how='outer', indicator=True)
outer

In [ ]:
# Diagnostic — see which rows matched
outer['_merge'].value_counts()

In [ ]:
# EXERCISE 3:
# Rate table: different expense load by line of business
rates = pd.DataFrame({
    'line': ['Auto', 'Property', 'Liability'],
    'expense_load': [0.30, 0.35, 0.25],
})

# Join rates onto df and create a new column 'pure_premium'
# pure_premium = premium * (1 - expense_load)
# YOUR CODE HERE

---
## Section 4 — Reshaping: pivot_table and melt

`pivot_table` → **wide format** (rows × columns matrix)  
`melt` → **long format** (one observation per row)

These are inverses of each other. Long format is better for plotting; wide format is better for reports.

In [ ]:
# Pivot: rows = line, columns = year, values = average loss ratio
pivot = df.pivot_table(
    values='loss_ratio',
    index='line',
    columns='year',
    aggfunc='mean',
)
pivot.round(3)

In [ ]:
# Add margins (row/column totals — uses aggfunc, so mean here)
pivot_m = df.pivot_table(
    values='loss_ratio',
    index='line',
    columns='year',
    aggfunc='mean',
    margins=True,
    margins_name='Overall',
)
pivot_m.round(3)

In [ ]:
# melt: convert the wide pivot back to long format
pivot_reset = pivot.reset_index()  # bring 'line' back as a column
long = pivot_reset.melt(
    id_vars='line',
    var_name='year',
    value_name='avg_loss_ratio',
)
long.sort_values(['line', 'year']).head(10)

In [ ]:
# EXERCISE 4:
# Create a pivot table showing TOTAL CLAIMS (sum) with
#   rows = company, columns = line, values = claims
# Then find which (company, line) cell has the highest claims
# YOUR CODE HERE

---
## Section 5 — String Methods and Categorical Dtype

`.str` accessor vectorizes string operations without loops.

| Method | Purpose |
|--------|---------|
| `.str.upper()` / `.lower()` | Case conversion |
| `.str.contains(pat, regex=True)` | Boolean mask |
| `.str.extract(pattern)` | Capture groups → columns |
| `.str.split(sep, expand=True)` | Split into columns |
| `.str.replace(pat, repl)` | Substitution |

**Categorical dtype** stores a column as integer codes + lookup table — saves memory and speeds up `groupby` on low-cardinality columns.

In [ ]:
# Work with a messy text column
claims_notes = pd.DataFrame({
    'claim_id': range(1, 9),
    'description': [
        'AUTO - rear-end collision, bodily injury',
        'PROPERTY - fire damage, total loss',
        'auto - side-swipe, minor',
        'LIABILITY - slip and fall',
        'Property - water damage',
        'AUTO - theft, comprehensive',
        'liability - dog bite',
        'AUTO - hail damage, comprehensive',
    ]
})

# Normalize to title case and extract line of business
claims_notes['desc_clean'] = claims_notes['description'].str.title()
claims_notes['lob'] = claims_notes['description'].str.extract(r'^([A-Za-z]+)')[0].str.title()
claims_notes

In [ ]:
# Filter to comprehensive claims
comp = claims_notes[claims_notes['description'].str.contains('comprehensive', case=False)]
comp

In [ ]:
# Convert 'line' in main df to categorical — compare memory usage
df['line_obj'] = df['line']  # object dtype copy
df['line_cat'] = df['line'].astype('category')

print(f"object dtype:      {df['line_obj'].memory_usage(deep=True):,} bytes")
print(f"category dtype:    {df['line_cat'].memory_usage(deep=True):,} bytes")

# Categories are ordered — useful for sorting
cat_type = pd.CategoricalDtype(['Auto', 'Property', 'Liability'], ordered=True)
df['line_ordered'] = df['line'].astype(cat_type)
df.sort_values('line_ordered')[['company','line_ordered','year']].head(10)

In [ ]:
# EXERCISE 5:
# Given descriptions below, extract both the lob AND the specific peril
# (the text after the dash) into separate columns using .str.extract
test = pd.Series([
    'AUTO - rear-end collision',
    'PROPERTY - fire damage',
    'LIABILITY - slip and fall',
])
# Hint: pattern r'^([A-Z]+) - (.+)'
# YOUR CODE HERE

---
## Section 6 — Time Series: Resampling and Rolling Windows

Requires a **DatetimeIndex**. Convert with `pd.to_datetime` and `set_index`.

| Operation | Syntax | What it does |
|-----------|--------|--------------|
| Resample | `df.resample('M').sum()` | Aggregate to calendar period |
| Rolling | `df.rolling(n).mean()` | Trailing n-period average |
| Expanding | `df.expanding().sum()` | Cumulative sum from start |
| Shift | `df.shift(n)` | Lag values by n periods |

**Common resample aliases:** `'D'` daily, `'W'` weekly, `'ME'` month-end, `'QE'` quarter-end, `'YE'` year-end

In [ ]:
# Simulate daily claims payments over 2 years
np.random.seed(7)
dates = pd.date_range('2022-01-01', '2023-12-31', freq='D')
ts = pd.DataFrame({
    'date': dates,
    'paid_claims': np.random.gamma(shape=2, scale=50_000, size=len(dates)).astype(int),
    'new_claims': np.random.poisson(lam=15, size=len(dates)),
}).set_index('date')

print(ts.shape)
ts.head()

In [ ]:
# Monthly totals
monthly = ts.resample('ME').sum()
monthly.head(6)

In [ ]:
# 30-day rolling average on daily data
ts['rolling_30d_paid'] = ts['paid_claims'].rolling(30).mean()

# Expanding (cumulative) total of new claims
ts['cumulative_new'] = ts['new_claims'].expanding().sum()

ts.head(35).tail(10)

In [ ]:
# Month-over-month change in paid claims
monthly['prev_month'] = monthly['paid_claims'].shift(1)
monthly['pct_change'] = monthly['paid_claims'].pct_change() * 100
monthly.round(1).head(8)

In [ ]:
# EXERCISE 6:
# a) Resample ts to quarterly totals (both columns)
# b) Compute a 90-day rolling MAX on paid_claims (different from mean — why might you care?)
# YOUR CODE HERE

---
## Section 7 — apply vs. Vectorized Operations

`apply` is flexible but slow — it runs Python in a loop. Prefer vectorized alternatives:

| Task | apply (slow) | Vectorized (fast) |
|------|-------------|-------------------|
| Conditional value | `apply(lambda r: ...)` | `np.where` / `pd.cut` |
| Math on column | `apply(math.sqrt)` | `np.sqrt(df['col'])` |
| String format | `apply(lambda x: f'{x:.1%}')` | `.map('{:.1%}'.format)` |
| Row-wise logic | `apply(func, axis=1)` | multiple column ops |

Use `apply` when no vectorized option exists (e.g. complex multi-column row logic or calling an external API).

In [ ]:
# BAD: apply for a simple condition
# df['flag'] = df['loss_ratio'].apply(lambda x: 'High' if x > 0.7 else 'OK')

# GOOD: np.where
df['flag'] = np.where(df['loss_ratio'] > 0.7, 'High', 'OK')

# GOOD: pd.cut for bins
df['lr_band'] = pd.cut(
    df['loss_ratio'],
    bins=[0, 0.5, 0.7, 0.9, 1.5],
    labels=['Low', 'Moderate', 'High', 'Extreme']
)

df[['company', 'line', 'year', 'loss_ratio', 'flag', 'lr_band']].head(10)

In [ ]:
# Timing comparison on a larger dataset
big = pd.DataFrame({'x': np.random.rand(200_000)})

%timeit big['x'].apply(lambda v: v ** 2)
%timeit big['x'] ** 2

In [ ]:
# Legitimate apply: complex row-level logic with multiple columns
def classify_risk(row):
    """Combine loss ratio and premium size into a risk tier."""
    if row['loss_ratio'] > 0.80 and row['premium'] < 1_000_000:
        return 'Concern'
    elif row['loss_ratio'] > 0.80:
        return 'Monitor'
    else:
        return 'Acceptable'

df['risk_tier'] = df.apply(classify_risk, axis=1)
df['risk_tier'].value_counts()

In [ ]:
# EXERCISE 7:
# Replace this apply with a vectorized equivalent using np.select or np.where
# tier_apply = df.apply(
#     lambda r: 'Large' if r['premium'] > 3_000_000
#               else ('Medium' if r['premium'] > 1_500_000 else 'Small'),
#     axis=1
# )
# YOUR CODE HERE — create df['size_tier'] without using apply

---
## Section 8 — Method Chaining with pipe

`pipe` lets you slot custom functions into a chain, keeping the readable top-to-bottom flow of `.groupby().agg().reset_index()` without intermediate variables.

```python
result = (
    raw_df
    .pipe(clean_columns)
    .pipe(add_features)
    .query('loss_ratio < 2')
    .groupby('line')
    .agg(...)
)
```

In [ ]:
def add_lr_flag(df, threshold=0.70):
    """Add a high-LR indicator column."""
    df = df.copy()
    df['high_lr'] = df['loss_ratio'] > threshold
    return df

def add_premium_tier(df):
    """Bucket premium into Small/Medium/Large."""
    df = df.copy()
    df['size'] = pd.cut(
        df['premium'],
        bins=[0, 1_500_000, 3_000_000, float('inf')],
        labels=['Small', 'Medium', 'Large'],
    )
    return df

result = (
    df[['company', 'line', 'year', 'premium', 'claims', 'loss_ratio']]
    .pipe(add_lr_flag, threshold=0.65)  # pass extra args after df
    .pipe(add_premium_tier)
    .query("year == 2023")
    .groupby(['line', 'size'], observed=True)
    .agg(count=('company','count'), avg_lr=('loss_ratio','mean'))
    .round(3)
)
result

In [ ]:
# EXERCISE 8 (capstone):
# Write a function normalize_lr(df) that:
#   - Computes z-score of loss_ratio within each line:  (lr - mean) / std
#   - Adds column 'lr_zscore'
# Then build a pipe chain that:
#   1. Applies normalize_lr
#   2. Filters to |lr_zscore| > 1 (outliers)
#   3. Sorts by lr_zscore descending
# YOUR CODE HERE

---
## Section 9 — Essential Data I/O

pandas reads and writes many formats. `read_csv` is the workhorse; `read_excel` handles spreadsheets.

| Parameter | `read_csv` effect |
|-----------|-------------------|
| `sep` | Delimiter (default `,`; `'\t'` for TSV) |
| `index_col` | Column(s) to use as the row index |
| `dtype` | Dict of column → dtype (avoids slow auto-detection) |
| `na_values` | Extra strings to treat as NaN |
| `nrows` | Read only the first N rows |
| `chunksize` | Return an iterator of DataFrames of size N |

Write functions mirror read functions: `to_csv`, `to_excel`, `to_json`.

In [ ]:
from io import StringIO
import tempfile, os

# Simulate a CSV with StringIO — no file needed for testing
csv_data = (
    'company,line,year,premium,claims
'
    'Alpha,Auto,2021,1200000,840000
'
    'Alpha,Property,2021,950000,570000
'
    'Beta,Auto,2021,1100000,715000
'
    'Beta,Liability,2021,800000,352000
'
    'Gamma,Property,2022,1350000,891000
'
)

df_io = pd.read_csv(
    StringIO(csv_data),
    dtype={'premium': 'float64', 'claims': 'float64'},
    index_col=['company', 'line'],
)
print('Shape:', df_io.shape)
print(df_io.head())

# Round-trip: write to file and reload
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, 'claims.csv')
    df_io.to_csv(path)
    reloaded = pd.read_csv(path, index_col=[0, 1])
    print('\nRound-trip match:', df_io.equals(reloaded))

# read_excel syntax reference (requires openpyxl)
print('\nread_excel: pd.read_excel(path, sheet_name=..., index_col=0)')

In [ ]:
from io import StringIO

# Chunked reading — memory-efficient for large files
big_csv = "id,loss\n" + "\n".join(f"{i},{i * 1000}" for i in range(1, 201))

chunk_sums = []
for chunk in pd.read_csv(StringIO(big_csv), chunksize=50):
    chunk_sums.append(chunk['loss'].sum())
print(f'4 chunks of 50 rows, total loss: {sum(chunk_sums):,}')

# JSON round-trip
df_small = pd.DataFrame({'line': ['Auto', 'Property'], 'lr': [0.68, 0.61]})
json_str = df_small.to_json(orient='records')
print('\nJSON:', json_str)

df_from_json = pd.read_json(StringIO(json_str))
print('Reloaded from JSON:')
print(df_from_json)

In [ ]:
# EXERCISE 9:
from io import StringIO
raw = (
    'accident_year,line,earned_premium,paid_losses
'
    '2021,Auto,1200000,840000
'
    '2021,Property,950000,589000
'
    '2022,Auto,1350000,
'
    '2022,Liability,880000,396000
'
    '2022,Property,1100000,671000
'
)
# a) Read raw into a DataFrame using StringIO; use accident_year as index; treat '' as NaN
# b) Print the dtypes and count of NaN values per column
# c) Write the DataFrame to a temp CSV, reload it, and confirm shape matches
# YOUR CODE HERE

---
## Section 10 — Missing Data and Cleaning

| Function | Purpose |
|----------|---------|
| `df.dropna(how, thresh, subset)` | Remove rows/cols with NaN |
| `df.fillna(value)` | Fill NaN with scalar, dict, or Series |
| `df.ffill()` / `df.bfill()` | Forward / backward fill along axis |
| `df.duplicated(subset, keep)` | Boolean mask of duplicate rows |
| `df.drop_duplicates(subset, keep)` | Remove duplicate rows |
| `df.replace(to_replace, value)` | Replace values (supports dicts and regex) |
| `df.rename(columns={...})` | Rename columns or index labels |

**`ffill()`** propagates the last valid value forward — ideal for sparse time series.

In [ ]:
# Build a DataFrame with intentional gaps
df_dirty = pd.DataFrame({
    'accident_year': [2021, 2021, 2022, 2022, 2023, 2023],
    'line':          ['Auto', 'Property', 'Auto', 'Liability', 'Property', 'Auto'],
    'premium':       [1_200_000, 950_000, 1_350_000, 880_000, None, 1_100_000],
    'claims':        [840_000, None, 972_000, None, 693_000, 726_000],
})
print('Missing counts:')
print(df_dirty.isna().sum(), '\n')

# dropna: different strategies
print('dropna(how="any"):', df_dirty.dropna(how='any').shape)
print('dropna(thresh=3): ', df_dirty.dropna(thresh=3).shape)     # keep rows with ≥3 non-NaN

# fillna with scalar vs forward-fill
df_zero = df_dirty.fillna(0)
df_ffill = df_dirty.ffill()    # propagates last valid value downward
print('\nfillna(0) premium:', df_zero['premium'].tolist())
print('ffill premium:     ', df_ffill['premium'].tolist())

# Group-mean fill: fill claims NaN with the mean for that line
df_gfill = df_dirty.copy()
df_gfill['claims'] = (df_dirty
    .groupby('line')['claims']
    .transform(lambda g: g.fillna(g.mean())))
print('\nGroup-mean fill claims:', df_gfill['claims'].round(0).tolist())

In [ ]:
# Duplicate detection and removal
df_dupes = pd.DataFrame({
    'claim_id': [1001, 1002, 1001, 1003, 1002],
    'severity':  [45_000, 12_000, 45_000, 280_000, 12_000],
    'status':    ['open', 'closed', 'open', 'open', 'closed'],
})
print('Total rows:', len(df_dupes))
print('Duplicate rows:', df_dupes.duplicated().sum())
print('Dupes on claim_id:', df_dupes.duplicated(subset='claim_id').sum())

df_clean = df_dupes.drop_duplicates(subset='claim_id', keep='first')
print('After drop_duplicates:', df_clean.shape)

# replace: recode categorical values
df_clean = df_clean.replace({'status': {'open': 1, 'closed': 0}})
print('\nStatus after replace:', df_clean['status'].tolist())

# rename: clean up column names
df_clean = df_clean.rename(columns={'claim_id': 'claim_id', 'severity': 'loss_amount', 'status': 'is_open'})
print('Columns:', df_clean.columns.tolist())

In [ ]:
# EXERCISE 10:
df_messy = pd.DataFrame({
    'Claim ID':      [201, 202, 203, 202, 204, 205],
    'Line of Bus.':  ['Auto', 'Property', None, 'Property', 'Auto', 'Liability'],
    'Severity ($)':  [32_000, None, 78_000, None, 15_000, 120_000],
    'Status':        ['Open', 'Closed', 'Open', 'Closed', 'Pend', 'Open'],
})
# a) Rename columns to snake_case: claim_id, line, severity, status
# b) Drop rows that are exact duplicates on claim_id (keep='first')
# c) Fill NaN severity with the median severity
# d) Replace 'Pend' in status with 'Pending'
# YOUR CODE HERE

---
## Section 11 — Sorting, Ranking, and Exploration

| Function | Purpose |
|----------|---------|
| `df.sort_values(by, ascending)` | Sort by one or more columns |
| `df.sort_index(level, ascending)` | Sort by index |
| `df.rank(method, ascending, pct)` | Assign ranks to values |
| `df.nlargest(n, col)` / `nsmallest` | Top/bottom N rows |
| `s.value_counts(normalize)` | Frequency table |
| `s.unique()` | Array of distinct values |
| `s.isin(values)` | Boolean membership mask |
| `df.corr()` / `df.cov()` | Pairwise correlation / covariance |
| `df.describe()` | Summary statistics |

`rank(method='min')` uses the lowest rank for ties; `method='average'` (default) splits them.

In [ ]:
df_flat = df.reset_index()

# sort_values by multiple columns
sorted_df = df_flat.sort_values(by=['line', 'loss_ratio'], ascending=[True, False])
print('Highest loss_ratio per line:')
print(sorted_df.groupby('line').first()[['company', 'year', 'loss_ratio']])

# nlargest / nsmallest
print('\nTop 5 premiums:')
print(df_flat.nlargest(5, 'premium')[['company', 'line', 'year', 'premium']])

# rank: which company had the most claims?
total_claims = df_flat.groupby('company')['claims'].sum()
print('\nTotal claims ranking (1 = highest):')
ranks = total_claims.rank(method='min', ascending=False).astype(int)
print(pd.concat([total_claims.rename('total_claims'), ranks.rename('rank')], axis=1))

In [ ]:
df_flat = df.reset_index()

# value_counts: distribution of a categorical column
print('Line distribution:')
print(df_flat['line'].value_counts(normalize=True).round(3))

# unique and isin
print('\nUnique companies:', df_flat['company'].unique())
subset = df_flat[df_flat['company'].isin(['Alpha', 'Gamma'])]
print(f'Rows for Alpha + Gamma: {len(subset)}')

# corr and cov
print('\nCorrelation matrix:')
print(df_flat[['premium', 'claims', 'loss_ratio']].corr().round(3))

# describe
print('\nDescriptive statistics:')
print(df_flat[['premium', 'claims', 'loss_ratio']].describe().round(0))

In [ ]:
# EXERCISE 11:
df_flat = df.reset_index()

# a) Sort df_flat by year ascending, then loss_ratio descending; print the top 5 rows
# b) Rank the 9 (company, line) combinations by mean premium — rank 1 = highest
#    Hint: groupby(['company','line'])['premium'].mean().rank(ascending=False)
# c) Use isin to filter to only ['Auto', 'Liability'] lines; print mean loss_ratio per line
# d) Compute the Pearson correlation between premium and claims; interpret the sign
# YOUR CODE HERE

---
## Section 12 — Binning, Dummies, and Sampling

| Function | Purpose |
|----------|---------|
| `pd.cut(s, bins, labels)` | Bin into custom-width intervals |
| `pd.qcut(s, q, labels)` | Bin into equal-frequency quantiles |
| `pd.get_dummies(df, columns, drop_first)` | One-hot encode categorical columns |
| `df.sample(n, frac, random_state)` | Random row sample |

`pd.cut` → custom break points (e.g., actuarial severity tiers).
`pd.qcut` → equal-count bins regardless of the value range.
`get_dummies(drop_first=True)` avoids the dummy variable trap in regression models.

In [ ]:
df_flat = df.reset_index()

# pd.cut: custom premium tiers
df_flat['premium_tier'] = pd.cut(
    df_flat['premium'],
    bins=[0, 800_000, 1_100_000, 1_500_000, float('inf')],
    labels=['Small', 'Mid', 'Large', 'XL'],
)
print('Premium tier counts:')
print(df_flat['premium_tier'].value_counts().sort_index())

# pd.qcut: equal-frequency loss_ratio quartiles
df_flat['lr_quartile'] = pd.qcut(df_flat['loss_ratio'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
print('\nLoss ratio quartile counts:')
print(df_flat['lr_quartile'].value_counts().sort_index())

# retbins=True shows the computed cut points
_, bins = pd.qcut(df_flat['loss_ratio'], q=4, retbins=True)
print('\nqcut breakpoints:', bins.round(3))

In [ ]:
df_flat = df.reset_index()

# One-hot encode 'line' — drop_first avoids perfect multicollinearity
df_enc = pd.get_dummies(
    df_flat[['company', 'line', 'year', 'premium', 'loss_ratio']],
    columns=['line'],
    drop_first=True,
)
print('Columns after get_dummies:', df_enc.columns.tolist())
print(df_enc.head(3))

# sample: reproducible random rows
sample_30pct = df_flat.sample(frac=0.30, random_state=42)
print(f'\n30% sample: {len(sample_30pct)} of {len(df_flat)} rows')

print('\n5-row sample:')
print(df_flat.sample(n=5, random_state=0)[['company', 'line', 'loss_ratio']])

In [ ]:
# EXERCISE 12:
df_flat = df.reset_index()

# a) Bin 'claims' into 3 equal-width tiers with pd.cut — label them 'Low', 'Mid', 'High'
# b) Bin 'premium' into 4 equal-frequency quantiles with pd.qcut — label Q1–Q4
# c) One-hot encode the 'company' column (drop_first=True); print the new column names
# d) Draw a stratified 1-row-per-line sample:
#    df_flat.groupby('line').sample(n=1, random_state=7)
# YOUR CODE HERE

---
## Section 13 — Concatenation and Advanced Reshaping

| Function | Purpose |
|----------|---------|
| `pd.concat([df1, df2], axis)` | Stack along rows (0) or columns (1) |
| `pd.concat(..., keys=[...])` | Add a hierarchical key level to the index |
| `df.stack()` | Move innermost column level → innermost row index |
| `df.unstack()` | Move innermost row index → innermost column level |
| `df1.combine_first(df2)` | Fill NaN in df1 with values from df2 |
| `df.swaplevel(0, 1)` | Swap two MultiIndex levels |

`stack` and `unstack` are inverses — they pivot between wide and long formats when the index is hierarchical.

In [ ]:
# Build three single-year DataFrames
year_dfs = {
    yr: pd.DataFrame({
        'line':    ['Auto', 'Property', 'Liability'],
        'premium': [1_200_000 + yr * 10_000, 950_000 + yr * 8_000, 800_000 + yr * 5_000],
        'claims':  [820_000 + yr * 7_000, 570_000 + yr * 4_000, 352_000 + yr * 3_000],
    })
    for yr in [2021, 2022, 2023]
}

# concat with keys → hierarchical index (year, row)
combined = pd.concat(year_dfs.values(), keys=year_dfs.keys(), names=['year', 'row'])
print('Combined shape:', combined.shape)
print(combined.head(6))

# combine_first: fill gaps in a sparse df with values from a fallback
base = pd.DataFrame({'lr': [0.68, None, 0.72]}, index=['Auto', 'Property', 'Liability'])
fallback = pd.DataFrame({'lr': [0.65, 0.60, 0.70]}, index=['Auto', 'Property', 'Liability'])
print('\nAfter combine_first:')
print(base.combine_first(fallback))

In [ ]:
df_flat = df.reset_index()

# Build a wide pivot (line × year loss_ratios)
wide = df_flat.pivot_table(values='loss_ratio', index='line', columns='year', aggfunc='mean')
print('Wide (line × year):')
print(wide.round(3))

# stack: column level (year) folds into row index → long format
long = wide.stack()
long.index.names = ['line', 'year']
print('\nAfter stack (long format):')
print(long.round(3).head(6))

# unstack: restore wide format
restored = long.unstack(level='year')
print('\nAfter unstack (wide again):')
print(restored.round(3))

# swaplevel on MultiIndex
df_multi = df_flat.set_index(['company', 'line'])
swapped = df_multi.swaplevel().sort_index()
print('\nswaplevel → (line, company) first rows:')
print(swapped.head(3))

In [ ]:
# EXERCISE 13:
ay21 = pd.DataFrame({'line': ['Auto', 'Property'], 'claims': [840_000, 570_000], 'year': 2021})
ay22 = pd.DataFrame({'line': ['Auto', 'Property'], 'claims': [862_000, 590_000], 'year': 2022})
ay23 = pd.DataFrame({'line': ['Auto', 'Property'], 'claims': [895_000, 612_000], 'year': 2023})

# a) Concatenate all three with keys=[2021,2022,2023]; name the index levels ['year','row']
# b) Pivot to wide: line rows × year columns, values=claims
# c) Stack the wide pivot to long; unstack back — confirm it matches the original pivot
# d) Build a sparse 'base' DataFrame with only Auto lr=0.68 (Property=NaN),
#    then use combine_first with a fallback that has both lines filled
# YOUR CODE HERE

---
## Section 14 — Advanced GroupBy Patterns

Beyond `agg` / `transform` / `filter` (Section 2), groupby supports:

| Pattern | Syntax | Use when |
|---------|--------|----------|
| Iterate groups | `for name, grp in df.groupby(...)` | Per-group diagnostics |
| Group by index level | `df.groupby(level='name')` | MultiIndex DataFrames |
| `ngroup()` | `df.groupby(...).ngroup()` | Numeric group ID per row |
| `pd.crosstab` | `pd.crosstab(rows, cols, values, aggfunc)` | Frequency / aggregated table |
| Quantile buckets | `groupby(pd.qcut(...))` | Stats within size/severity bands |

In [ ]:
df_flat = df.reset_index()

# Iterate over groups — useful for per-group diagnostics
print('=== Loss ratio summary by company ===')
for company, grp in df_flat.groupby('company'):
    mean_lr = grp['loss_ratio'].mean()
    max_lr  = grp['loss_ratio'].max()
    print(f'  {company:8s}  mean LR: {mean_lr:.3f}   max LR: {max_lr:.3f}')

# Groupby on MultiIndex index level (no reset_index needed)
print('\nMean premium by company (index level):')
print(df.groupby(level='company')['premium'].mean().round(0))

# ngroup: sequential integer group ID
df_flat['company_id'] = df_flat.groupby('company').ngroup()
print('\nCompany → group ID:')
print(df_flat[['company', 'company_id']].drop_duplicates().reset_index(drop=True))

In [ ]:
df_flat = df.reset_index()

# pd.crosstab: frequency table
freq = pd.crosstab(df_flat['company'], df_flat['line'])
print('Frequency (company × line):')
print(freq)

# Aggregated crosstab: mean loss_ratio per cell
lr_cross = pd.crosstab(
    df_flat['company'], df_flat['line'],
    values=df_flat['loss_ratio'], aggfunc='mean',
)
print('\nMean loss_ratio (company × line):')
print(lr_cross.round(3))

# Quantile groups: stats within premium size bands
df_flat['prem_band'] = pd.qcut(df_flat['premium'], q=3, labels=['Small', 'Mid', 'Large'])
print('\nMean loss_ratio by premium band:')
print(df_flat.groupby('prem_band', observed=True)['loss_ratio'].agg(['mean', 'count']))

In [ ]:
# EXERCISE 14:
df_flat = df.reset_index()

# a) Iterate over groupby('line') and print, for each line,
#    the company with the highest average loss_ratio in that line
# b) Use pd.crosstab with values=claims and aggfunc='sum' to produce
#    a company × year claims table
# c) Group df_flat by pd.qcut(df_flat['claims'], q=3, labels=['Low','Mid','High'])
#    and compute mean premium in each bucket
# YOUR CODE HERE

---
## Section 15 — Time Series: Datetime Parsing and EWM

| Function / Method | Purpose |
|-------------------|---------|
| `pd.to_datetime(s)` | Parse strings / mixed formats → `DatetimeIndex` |
| `pd.Timestamp('2022-01-31')` | Single datetime object |
| `pd.date_range(start, periods, freq)` | Generate a sequence of dates |
| `series.dt.year` / `.month` / `.quarter` | Extract datetime components |
| `ts.ewm(span=n).mean()` | Exponentially weighted moving average |
| `ts.resample('W').interpolate()` | Upsample then fill gaps |
| `ts.tz_localize('UTC').tz_convert(tz)` | Time zone conversion |

`ewm(span=N)` assigns exponentially decaying weights — recent values count more than older ones, smoothing volatile claims data without a hard window cutoff.

In [ ]:
# Parse a mix of date strings
dates_raw = ['2022-01-15', '2022-02-28', 'March 31 2022', '2022-04-30']
ts_idx = pd.to_datetime(dates_raw)
print('Parsed DatetimeIndex:', ts_idx)
print('Month:', ts_idx.month.tolist())
print('Day of week (0=Mon):', ts_idx.dayofweek.tolist())

# pd.Timestamp — single datetime with useful attributes
t = pd.Timestamp('2022-06-30')
print(f'\nTimestamp: {t}  quarter: {t.quarter}  days_in_month: {t.days_in_month}')

# dt accessor on a Series
df_dates = pd.DataFrame({
    'report_date': pd.date_range('2022-01', periods=12, freq='ME'),
    'claims':      [820, 740, 890, 810, 950, 1020, 870, 900, 960, 880, 820, 1100],
})
df_dates['month']   = df_dates['report_date'].dt.month
df_dates['quarter'] = df_dates['report_date'].dt.quarter
print('\nWith extracted date parts:')
print(df_dates.head(4))

In [ ]:
# Monthly claims series
idx = pd.date_range('2022-01', periods=24, freq='ME')
rng = np.random.default_rng(42)
monthly = pd.Series(rng.integers(700, 1200, size=24).astype(float), index=idx, name='claims')

# Rolling vs EWM — EWM weights recent data more heavily
comp = pd.DataFrame({
    'claims':     monthly,
    'rolling_12': monthly.rolling(12).mean(),
    'ewm_12':     monthly.ewm(span=12).mean(),
})
print('Rolling vs EWM (last 6 months):')
print(comp.tail(6).round(1))

# Upsampling: monthly → weekly via linear interpolation
weekly = monthly.resample('W').interpolate(method='linear')
print(f'\nMonthly: {len(monthly)} points → weekly (interpolated): {len(weekly)} points')

# Time zone localization and conversion
utc_idx = pd.date_range('2022-01-01', periods=3, freq='ME', tz='UTC')
eastern = utc_idx.tz_convert('America/New_York')
print('\nUTC to Eastern:')
for u, e in zip(utc_idx, eastern):
    print(f'  {u}  →  {e}')

In [ ]:
# EXERCISE 15:
date_strings = ['2021-03-31', '2021-06-30', '2021-09-30', '2021-12-31',
                '2022-03-31', '2022-06-30', '2022-09-30', '2022-12-31']
claims_vals  = [820, 940, 870, 1050, 900, 980, 920, 1100]

# a) Convert date_strings to a DatetimeIndex and build a quarterly claims Series
# b) Extract year and quarter from the index using .dt.year and .dt.quarter
# c) Compute the 2-quarter EWM (span=2) of claims
# d) Upsample the quarterly series to monthly with .resample('ME').interpolate('linear')
# YOUR CODE HERE

---
## Section 16 — pandas Plotting

pandas wraps matplotlib through a `.plot()` method on Series and DataFrames.

| Kind | Syntax | Use when |
|------|--------|----------|
| Line | `df.plot(figsize, title)` | Trends over time |
| Bar | `df.plot.bar(stacked)` | Categorical comparisons |
| Horizontal bar | `df.plot.barh()` | Many categories |
| Histogram | `s.plot.hist(bins, alpha)` | Distribution of values |
| Scatter | `df.plot.scatter(x, y)` | Relationship between two variables |
| Subplots | `df.plot(subplots=True, layout=(r,c))` | Multiple panels in one figure |

**Tip:** Call `plt.tight_layout()` to prevent label overlap; use `plt.savefig('out.png')` to save.

In [ ]:
import matplotlib.pyplot as plt

# Line plot: monthly claims trends
idx = pd.date_range('2022-01', periods=24, freq='ME')
rng = np.random.default_rng(42)
ts_plot = pd.DataFrame({
    'paid_claims': rng.integers(700, 1200, 24),
    'new_claims':  rng.integers(400,  800, 24),
}, index=idx)

ax = ts_plot.plot(figsize=(9, 3), title='Monthly Claims (2022-2023)')
ax.set_ylabel('Claims ($000)')
plt.tight_layout()
plt.show()

# Bar plot: total premium by line of business
df_flat = df.reset_index()
prem_by_line = df_flat.groupby('line')['premium'].sum() / 1e6
ax2 = prem_by_line.plot.bar(figsize=(5, 3), title='Total Premium by Line ($M)', color='steelblue', rot=0)
ax2.set_ylabel('Premium ($M)')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

df_flat = df.reset_index()

# Histogram: loss ratio distribution
ax = df_flat['loss_ratio'].plot.hist(
    bins=12, alpha=0.7, figsize=(5, 3), title='Loss Ratio Distribution', color='tomato'
)
ax.set_xlabel('Loss Ratio')
plt.tight_layout()
plt.show()

# Scatter: premium vs claims, coloured by line
colors = {'Auto': 'steelblue', 'Property': 'darkorange', 'Liability': 'seagreen'}
fig, ax = plt.subplots(figsize=(5, 4))
for line, grp in df_flat.groupby('line'):
    ax.scatter(grp['premium'] / 1e6, grp['claims'] / 1e6, label=line, color=colors[line], s=60)
ax.set_xlabel('Premium ($M)')
ax.set_ylabel('Claims ($M)')
ax.set_title('Premium vs Claims by Line')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# EXERCISE 16:
import matplotlib.pyplot as plt
df_flat = df.reset_index()

# a) Stacked bar chart of total claims by line (rows) and year (stacked columns):
#    Hint: pivot_table(values='claims', index='line', columns='year', aggfunc='sum')
#          .plot.bar(stacked=True, title='Claims by Line and Year')
# b) Histogram of 'premium' with 8 bins, title 'Premium Distribution'
# c) Scatter plot: loss_ratio (x) vs claims (y), each company a different colour
#    Hint: loop over groupby('company') and call ax.scatter for each group
# YOUR CODE HERE

---
## Wrap-Up Cheat Sheet

| Topic | Key takeaway |
|-------|--------------|
| MultiIndex | Use `.xs(key, level=name)` for inner-level slices |
| groupby | `agg` shrinks, `transform` preserves shape, `filter` drops groups |
| merge | Always check with `indicator=True`; unmatched rows become NaN |
| reshape | `pivot_table` wide; `melt` long; they're inverses |
| strings | `.str.extract(r'pattern')` for structured text parsing |
| category | Low-cardinality columns: `astype('category')` saves memory |
| time series | `resample` aggregates; `rolling`/`expanding` slide windows |
| apply | Last resort — prefer `np.where`, `pd.cut`, `np.select` |
| pipe | Keeps multi-step transforms readable as a single chain |
| Data I/O | `read_csv`/`to_csv`; `chunksize` for large files; `read_json` |
| Missing data | `dropna` removes; `fillna`/`ffill` fills; `replace` recodes |
| Sort / rank | `sort_values(by=[...], ascending=[...])` then `.rank()` |
| Binning | `pd.cut` = custom widths; `pd.qcut` = equal counts; `get_dummies` for ML |
| concat / stack | `pd.concat` joins DataFrames; `stack`/`unstack` pivot hierarchical indexes |
| Advanced groupby | `pd.crosstab` for frequency tables; iterate groups for diagnostics |
| Datetime / EWM | `pd.to_datetime` parses; `ewm(span)` weights recent values more |
| Plotting | `.plot.bar(stacked)`, `.plot.hist(bins)`, `.plot.scatter(x,y)` |